# 03 — Linear Probing
Freeze all layers, train only LM head. GPU needed.

In [ ]:
import os, sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()

if IS_KAGGLE:
    REPO_URL = "https://github.com/komalpreet10/medical-llm-peft-benchmark"
    REPO_DIR = Path("/kaggle/working/medical-llm-peft-benchmark")

    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}

    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))

    from src.setup import setup_kaggle
    setup_kaggle()
else:
    sys.path.insert(0, "..")

    from src.setup import setup_local
    setup_local()


In [ ]:
from src.data_utils  import get_tokenizer, load_train_data, load_test_data
from src.model_utils import load_base_model, apply_linear_probing, count_parameters
from src.train_utils import init_trackers, get_training_args, get_trainer, log_metrics, end_trackers
from src.eval_utils  import evaluate_model, save_metrics

init_trackers("linear_probing")

In [ ]:
tokenizer  = get_tokenizer()
train_data = load_train_data(tokenizer)
test_data  = load_test_data()

In [ ]:
model  = load_base_model()
model  = apply_linear_probing(model)
params = count_parameters(model)
print(params)

In [ ]:
args    = get_training_args("./outputs/linear_probing")
trainer = get_trainer(model, tokenizer, train_data, args)
trainer.train()
trainer.save_model("./outputs/linear_probing/final")

In [ ]:
metrics = evaluate_model(model, tokenizer, test_data, "linear_probing")
metrics.update(params)
save_metrics(metrics, "results/linear_probing_metrics.json")

In [ ]:
log_metrics(metrics)
end_trackers()
print("Linear Probing complete")